# 04 — VLM Perception Comparison

Compares Vision Language Models for understanding Barcelona Eixample street environments:
- **PerceptionLM** (Meta, local)
- **LLaVA 1.6** (Ollama, local)
- **GPT-4o-Vision** (OpenAI API)
- **MiniCPM-V** (Ollama, local)

**Task**: Given street view images, each VLM answers:
1. What type of street environment? (commercial / residential / mixed / park)
2. What amenities are visible?
3. Rate appeal for each archetype (tourist / resident / commuter) 1-5

**Metrics**: Classification accuracy, amenity recall F1, latency, archetype alignment

In [2]:
import asyncio
import base64
import json
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import AsyncOpenAI

sns.set_theme(style='whitegrid', palette='muted')

# Ground truth labels for test images (add your images to data/images/)
GROUND_TRUTH = [
    {'image': 'data/images/eixample_commercial_01.jpg', 'env_type': 'commercial',
     'amenities': ['cafe', 'shop', 'restaurant'], 'tourist_score': 4, 'resident_score': 3, 'commuter_score': 3},
    {'image': 'data/images/eixample_residential_01.jpg', 'env_type': 'residential',
     'amenities': ['pharmacy', 'supermarket'], 'tourist_score': 2, 'resident_score': 5, 'commuter_score': 4},
    {'image': 'data/images/eixample_mixed_01.jpg', 'env_type': 'mixed',
     'amenities': ['cafe', 'pharmacy'], 'tourist_score': 3, 'resident_score': 4, 'commuter_score': 4},
    {'image': 'data/images/eixample_park_01.jpg', 'env_type': 'park',
     'amenities': ['park'], 'tourist_score': 5, 'resident_score': 4, 'commuter_score': 2},
    {'image': 'data/images/eixample_commercial_02.jpg', 'env_type': 'commercial',
     'amenities': ['restaurant', 'bar', 'shop'], 'tourist_score': 5, 'resident_score': 3, 'commuter_score': 2},
]

VLM_PROVIDERS = {
    'llava-ollama': {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'llava:13b'},
    'minicpm-ollama': {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'minicpm-v'},
    'gpt4o-vision': {'base_url': None, 'api_key': os.getenv('OPENAI_API_KEY', ''), 'model': 'gpt-4o'},
}
# PerceptionLM: runs locally via transformers (not OpenAI-compatible API)
# Handled separately in the PLM benchmark cell below
PLM_MODEL_PATH = 'facebook/Perception-LM-1B'

print('VLM providers configured. Add images to data/images/ before running.')

VLM providers configured. Add images to data/images/ before running.


In [ ]:
VISION_PROMPT = """Analyse this street view image from Barcelona Eixample.
Reply with JSON only:
{
  \"env_type\": \"commercial|residential|mixed|park\",
  \"amenities_visible\": [\"list\", \"of\", \"amenity\", \"types\"],
  \"tourist_appeal\": 1-5,
  \"resident_appeal\": 1-5,
  \"commuter_appeal\": 1-5,
  \"reasoning\": \"brief explanation\"
}"""

def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

async def query_vlm(provider_name, config, image_path):
    if not os.path.exists(image_path):
        return None, 0, False
    img_b64 = encode_image(image_path)
    kwargs = {'api_key': config['api_key']}
    if config['base_url']:
        kwargs['base_url'] = config['base_url']
    client = AsyncOpenAI(**kwargs)
    t0 = time.perf_counter()
    try:
        resp = await client.chat.completions.create(
            model=config['model'],
            messages=[{'role': 'user', 'content': [
                {'type': 'text', 'text': VISION_PROMPT},
                {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}}
            ]}],
            max_tokens=300
        )
        elapsed = (time.perf_counter() - t0) * 1000
        content = resp.choices[0].message.content or ''
        parsed = json.loads(content)
        return parsed, elapsed, True
    except Exception as e:
        elapsed = (time.perf_counter() - t0) * 1000
        return None, elapsed, False

print('Run async benchmark cells below with actual images present')

Run async benchmark cells below with actual images present


: 

## PerceptionLM Local Inference

PerceptionLM runs locally via HuggingFace `transformers` (not through an OpenAI-compatible API).
The cell below loads the model and provides a `query_plm()` function that matches the
same interface as `query_vlm()` for fair benchmarking.

In [ ]:
import re
import torch
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText

def load_plm(model_path=PLM_MODEL_PATH):
    """Load PerceptionLM-1B for local inference."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32
    processor = AutoProcessor.from_pretrained(model_path, use_fast=True)
    model = AutoModelForImageTextToText.from_pretrained(
        model_path, dtype=dtype
    ).to(device)
    model.eval()
    return processor, model, device

def query_plm(processor, model, device, image_path):
    """Query PerceptionLM with VISION_PROMPT. Returns (parsed, latency_ms, success)."""
    if not os.path.exists(image_path):
        return None, 0, False
    image_path_str = str(Path(image_path).resolve())
    conversation = [
        {"role": "user", "content": [
            {"type": "image", "url": image_path_str},
            {"type": "text", "text": VISION_PROMPT},
        ]}
    ]
    inputs = processor.apply_chat_template(
        [conversation], add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    t0 = time.perf_counter()
    try:
        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=300)
        elapsed = (time.perf_counter() - t0) * 1000
        input_len = inputs["input_ids"].shape[1]
        raw = processor.batch_decode(gen_ids[:, input_len:], skip_special_tokens=True)[0]
        match = re.search(r"\{[\s\S]*\}", raw)
        parsed = json.loads(match.group()) if match else json.loads(raw)
        return parsed, elapsed, True
    except Exception as e:
        elapsed = (time.perf_counter() - t0) * 1000
        print(f"  PLM error: {e}")
        return None, elapsed, False

# Load model (takes ~30s on CPU)
plm_processor, plm_model, plm_device = load_plm()
print(f"PerceptionLM loaded on {plm_device}")

In [ ]:
# Full benchmark: run all providers + PerceptionLM on ground-truth images
async def run_full_benchmark():
    all_records = []
    for gt in GROUND_TRUTH:
        img = gt['image']
        print(f'\nImage: {img}')
        # OpenAI-compatible providers
        for pname, cfg in VLM_PROVIDERS.items():
            parsed, lat, ok = await query_vlm(pname, cfg, img)
            env_correct = (parsed or {}).get('env_type', '') == gt['env_type'] if ok else False
            pred_amenities = set((parsed or {}).get('amenities_visible', []))
            gt_amenities = set(gt['amenities'])
            tp = len(pred_amenities & gt_amenities)
            prec = tp / len(pred_amenities) if pred_amenities else 0
            rec = tp / len(gt_amenities) if gt_amenities else 0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
            all_records.append({'provider': pname, 'image': img, 'latency_ms': lat,
                                'env_correct': env_correct, 'amenity_f1': f1, 'success': ok})
            print(f'  {pname}: {lat:.0f}ms, env={env_correct}, f1={f1:.2f}')
        # PerceptionLM (local, synchronous)
        parsed, lat, ok = query_plm(plm_processor, plm_model, plm_device, img)
        env_correct = (parsed or {}).get('env_type', '') == gt['env_type'] if ok else False
        pred_amenities = set((parsed or {}).get('amenities_visible', []))
        gt_amenities = set(gt['amenities'])
        tp = len(pred_amenities & gt_amenities)
        prec = tp / len(pred_amenities) if pred_amenities else 0
        rec = tp / len(gt_amenities) if gt_amenities else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        all_records.append({'provider': 'perceptionlm', 'image': img, 'latency_ms': lat,
                            'env_correct': env_correct, 'amenity_f1': f1, 'success': ok})
        print(f'  perceptionlm: {lat:.0f}ms, env={env_correct}, f1={f1:.2f}')
    return pd.DataFrame(all_records)

# Uncomment to run live benchmark (requires images + running services):
# benchmark_df = await run_full_benchmark()
print('Benchmark function ready. Uncomment the line above to run with real images.')

In [ ]:
# Placeholder results for offline viewing (replace with actual benchmark run)
import numpy as np
np.random.seed(42)

records = []
for provider, lat_mean, acc in [('llava-ollama', 4200, 0.60), ('minicpm-ollama', 2800, 0.65),
                                  ('gpt4o-vision', 1100, 0.88), ('perceptionlm', 900, 0.82)]:
    for _ in range(10):
        records.append({'provider': provider, 'latency_ms': max(100, lat_mean + np.random.randn()*lat_mean*0.2),
                        'env_correct': np.random.rand() < acc, 'amenity_f1': acc + np.random.randn()*0.1})

df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Latency
sns.violinplot(data=df, x='provider', y='latency_ms', ax=axes[0], inner='box')
axes[0].set_title('VLM Response Latency')
axes[0].set_ylabel('Latency (ms)')
axes[0].tick_params(axis='x', rotation=20)

# Accuracy
acc_df = df.groupby('provider')[['env_correct', 'amenity_f1']].mean()
acc_df.plot(kind='bar', ax=axes[1])
axes[1].set_title('Classification Accuracy & Amenity F1')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=20)

# Scatter
summary = df.groupby('provider').agg({'latency_ms': 'median', 'amenity_f1': 'mean'}).reset_index()
axes[2].scatter(summary['latency_ms'], summary['amenity_f1'], s=120)
for _, row in summary.iterrows():
    axes[2].annotate(row['provider'], (row['latency_ms'], row['amenity_f1']),
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[2].set_title('Latency vs Amenity F1')
axes[2].set_xlabel('Median Latency (ms)')
axes[2].set_ylabel('Amenity F1')

plt.tight_layout()
plt.savefig('results_04_vlm_comparison.png', dpi=150)
plt.show()